**Legacy notebook.** Self-contained analysis code that predates the `src/mrvf` library and has not been ported to it. Kept for provenance and because it still produces figures in `results/`. Paths were updated to the `results/` layout; the next cell sets the working directory to the repository root, so run it from anywhere.

For the maintained pipeline see `notebooks/01_train_triple_regime.ipynb` and `notebooks/02_evaluate_rmse_vs_snr.ipynb`.

In [ ]:
import os
from pathlib import Path
# run from the repository root so ./results/... and ../subsamples resolve
_root = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "mrvf").is_dir())
os.chdir(_root)

# T1w Brain Image — Same Slices as Parameter Maps

Loads `e11_T1_brain.nii.gz` and displays the same 4 slices used in the
`final_InVivo_SmoothFirst_v1` parameter-map figure (`SUBJECT_KEY = 'img_e11air'`,
`N_SLICES = 4`, `SLICE_OFFSET = 3`).  
Slice selection mirrors the original logic: pick slices with brain coverage
> 30 % of peak, skip the first `SLICE_OFFSET` good slices, then take
every `step`-th slice for a total of `N_SLICES`.

The slices confirmed in the original notebook output were **[3, 5, 7, 9]**.
If the T1 has a different z-dimension the cell below will adapt automatically.

In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family'   : 'Arial',
    'font.size'     : 9,
    'figure.dpi'    : 150,
    'savefig.dpi'   : 300,
    'savefig.bbox'  : 'tight',
})

In [ ]:
# ── Configuration — mirror final_InVivo_SmoothFirst_v1 ─────────────────────
T1_PATH     = '../GESFIDE_data/e11_T1_brain.nii.gz'
SUBJECT_ID  = 'E11'
CONDITION   = 'air'

N_SLICES     = 4    # slices to show
SLICE_OFFSET = 3    # skip first N good slices (edge/noisy)

# These are the slices confirmed in the original notebook output.
# Leave as None to re-derive from the T1 brain coverage.
FORCE_SLICES = None   # e.g. [3, 5, 7, 9] to hard-code

In [ ]:
# ── Load T1w image ──────────────────────────────────────────────────────────
img   = nib.load(T1_PATH)
data  = img.get_fdata().astype(np.float32)   # (H, W, S) or (H, W, S, 1)
if data.ndim == 4:
    data = data[..., 0]

H, W, S = data.shape
print(f'T1w shape  : {H} x {W} x {S}')
print(f'Voxel size : {img.header.get_zooms()}')
print(f'Intensity  : min={data.min():.1f}  max={data.max():.1f}')

In [ ]:
# ── Slice selection (same logic as original notebook) ───────────────────────
if FORCE_SLICES is not None:
    show_slices = np.array(FORCE_SLICES)
    print(f'Using forced slices : {show_slices.tolist()}')
else:
    # Brain mask: any non-zero voxel
    brain_mask     = data > 0
    brain_coverage = brain_mask.sum(axis=(0, 1))   # (S,)
    good_slices    = np.where(brain_coverage > brain_coverage.max() * 0.3)[0]
    good_slices    = good_slices[SLICE_OFFSET:]     # drop leading edge slices
    step           = max(len(good_slices) // N_SLICES, 1)
    show_slices    = good_slices[::step][:N_SLICES]
    print(f'Brain coverage : {brain_coverage.tolist()}')
    print(f'Good slices    : {good_slices.tolist()}')
    print(f'Selected slices: {show_slices.tolist()}')

In [ ]:
from nilearn.image import resample_to_img
import nibabel as nib

# Load T1 in native space
t1_native = nib.load(T1_PATH)

# Use the GM mask (already in GESFIDE space) as the resampling reference
mask_ref_path = '../GESFIDE_data/GES_ROI/E11_AIR_ROI.nii.gz'  # adjust filename if needed
mask_ref = nib.load(mask_ref_path)

# Resample T1 → GESFIDE space (128×128×14)
t1_resampled = resample_to_img(t1_native, mask_ref, interpolation='continuous')
data = t1_resampled.get_fdata().astype(np.float32)
if data.ndim == 4:
    data = data[..., 0]

H, W, S = data.shape
print(f'T1w resampled shape: {H} x {W} x {S}')   # should print 128 x 128 x 14

In [ ]:
show_slices = np.array([3, 5, 7, 9])   # confirmed from original notebook output
print(f'Using GESFIDE-matched slices: {show_slices.tolist()}')

In [ ]:
# ── Plot ────────────────────────────────────────────────────────────────────
# Intensity clip: 1st–99th percentile of brain voxels for good contrast
brain_vals = data[data > 0]
vmin = float(np.percentile(brain_vals, 1))
vmax = float(np.percentile(brain_vals, 99))

fig_w = N_SLICES * 2.2 + 0.4
fig_h = 2.4

fig, axes = plt.subplots(
    1, N_SLICES,
    figsize=(fig_w, fig_h),
    gridspec_kw={'wspace': 0.03}
)
fig.patch.set_facecolor('black')
fig.suptitle(
    f'Subject {SUBJECT_ID}  |  {CONDITION}  |  T1w brain',
    fontsize=10, fontweight='bold', color='white', y=1.02
)

for si, sl in enumerate(show_slices):
    ax = axes[si]
    ax.set_facecolor('black')

    # Mask out zero background so it stays black
    sl_data = data[:, :, sl].copy()
    sl_ma   = np.ma.masked_where(sl_data == 0, sl_data)

    cmap = plt.colormaps['gray'].copy()
    cmap.set_bad('black')

    ax.imshow(
        np.rot90(sl_ma),
        cmap=cmap,
        vmin=vmin, vmax=vmax,
        interpolation='nearest'
    )
    ax.set_title(f'sl {sl}', fontsize=7, color='#888888', pad=2)
    ax.axis('off')

plt.tight_layout(pad=0.3)
fig.savefig('results/figures/t1w_slices_e11.png', dpi=300, bbox_inches='tight',
            facecolor='black')
plt.show()
print('Saved: results/figures/t1w_slices_e11.png')

In [ ]:
import scipy.io as sio, h5py

def load_mat_any(path, key):
    try:
        return np.array(sio.loadmat(path)[key], dtype=np.float32)
    except NotImplementedError:
        with h5py.File(path, 'r') as f:
            arr = np.array(f[key][()])
            return arr.T.astype(np.float32)

COND_PATHS = {
    'air'  : './results/invivo_smooth_v1/img_e11air_maps_smooth.mat',
    'hyper': './results/invivo_smooth_v1/img_e11hyper_maps_smooth.mat',
    'hypo' : './results/invivo_smooth_v1/img_e11hypo_maps_smooth.mat',
}
METHOD_KEY = 'Triple'

cond_maps = {}
gm_mask   = None
for cond, path in COND_PATHS.items():
    cond_maps[cond] = load_mat_any(path, METHOD_KEY)
    if gm_mask is None:
        gm_mask = load_mat_any(path, 'mask').astype(bool)
    print(f'{cond}: {cond_maps[cond].shape}')

In [ ]:
SHOW_SLICES = [3, 5, 7, 9]
PARAM_VIS = [
    ('SO2 (%)',  0, 100,   (50, 100), 'hot'),
    ('CBV (%)',  1, 100,   (2,   8),  'turbo'),
    ('R (um)',   2, 1e6,   (15,  28), 'pink'),
    ('T2 (ms)',  3, 1000,  (40, 120), 'bone'),
]
CONDITIONS  = list(COND_PATHS.keys())
METHOD_KEYS = ['Triple', 'DM']

for plabel, pidx, scale, (vmin, vmax), cmap_n in PARAM_VIS:
    cmap_obj = plt.colormaps[cmap_n].copy()
    cmap_obj.set_bad('black')

    n_rows = len(CONDITIONS) * len(METHOD_KEYS)   # 6 rows total
    fig, axes = plt.subplots(
        n_rows, len(SHOW_SLICES),
        figsize=(len(SHOW_SLICES) * 2.2, n_rows * 2.2),
        gridspec_kw={'hspace': 0.04, 'wspace': 0.03}
    )
    fig.patch.set_facecolor('black')

    for ri, (cond, mkey) in enumerate(
            [(c, m) for c in CONDITIONS for m in METHOD_KEYS]):
        maps = load_mat_any(COND_PATHS[cond], mkey)
        axes[ri, 0].set_ylabel(f'{cond}\n{mkey}', color='white', fontsize=8,
                               rotation=0, labelpad=48, va='center')
        for si, sl in enumerate(SHOW_SLICES):
            ax = axes[ri, si]
            ax.set_facecolor('black')
            img = np.ma.array(maps[:, :, sl, pidx] * scale,
                              mask=~gm_mask[:, :, sl])
            ax.imshow(np.rot90(img), cmap=cmap_obj,
                      vmin=vmin, vmax=vmax, interpolation='nearest')
            ax.axis('off')

    plt.suptitle(f'E11  —  {plabel}', color='white',
                 fontsize=11, fontweight='bold', y=1.02)
    plt.savefig(f'results/figures/e11_{plabel.split()[0].lower()}_conditions.png',
                dpi=300, bbox_inches='tight', facecolor='black')
    plt.show()

In [ ]:
METHODS = [('DM', 'DM'), ('Triple', 'DL')]
CONDITIONS = ['air', 'hyper', 'hypo']
PARAM_VIS_ORDERED = [
    ('CBV (%)',  1, 100,   (2,   8),  'turbo'),
    ('SO2 (%)',  0, 100,   (50, 100), 'hot'),
    ('R (um)',   2, 1e6,   (15,  28), 'pink'),
    ('T2 (ms)',  3, 1000,  (40, 120), 'bone'),
]
SHOW_SLICES = [3]

N_SL   = len(SHOW_SLICES)
N_COND = len(CONDITIONS)
N_METH = len(METHODS)
N_COLS = N_METH * N_COND * N_SL   # 24

fig, axes = plt.subplots(
    len(PARAM_VIS_ORDERED), N_COLS,
    figsize=(N_COLS * 1.4, len(PARAM_VIS_ORDERED) * 2.0),
    gridspec_kw={'hspace': 0.04, 'wspace': 0.02}
)
fig.patch.set_facecolor('black')

for ri, (plabel, pidx, scale, (vmin, vmax), cmap_n) in enumerate(PARAM_VIS_ORDERED):
    cmap_obj = plt.colormaps[cmap_n].copy()
    cmap_obj.set_bad('black')

    for mi, (mkey, _) in enumerate(METHODS):
        for ci, cond in enumerate(CONDITIONS):
            maps = load_mat_any(COND_PATHS[cond], mkey)
            for si, sl in enumerate(SHOW_SLICES):
                col = mi * N_COND * N_SL + ci * N_SL + si
                ax  = axes[ri, col]
                ax.set_facecolor('black')
                img = np.ma.array(maps[:, :, sl, pidx] * scale,
                                  mask=~gm_mask[:, :, sl])
                ax.imshow(np.rot90(img), cmap=cmap_obj,
                          vmin=vmin, vmax=vmax, interpolation='nearest')
                ax.axis('off')

    axes[ri, 0].set_ylabel(plabel, color='white', fontsize=8,
                           rotation=90, labelpad=4, va='center')

# ── Headers: method (top) and condition (below) ──────────────────────────────
fig.canvas.draw()   # needed to resolve axes positions

for mi, (_, mlabel) in enumerate(METHODS):
    for ci, cond in enumerate(CONDITIONS):
        col0 = mi * N_COND * N_SL + ci * N_SL
        col1 = col0 + N_SL - 1
        xL = axes[0, col0].get_position().x0
        xR = axes[0, col1].get_position().x1
        xm = (xL + xR) / 2
        y_cond = axes[0, 0].get_position().y1 + 0.01
        fig.text(xm, y_cond, cond, ha='center', va='bottom',
                 color='#aaaaaa', fontsize=8, transform=fig.transFigure)

    col0 = mi * N_COND * N_SL
    col1 = col0 + N_COND * N_SL - 1
    xL = axes[0, col0].get_position().x0
    xR = axes[0, col1].get_position().x1
    xm = (xL + xR) / 2
    y_meth = axes[0, 0].get_position().y1 + 0.045
    fig.text(xm, y_meth, mlabel, ha='center', va='bottom',
             color='white', fontsize=10, fontweight='bold',
             transform=fig.transFigure)

# ── Vertical divider between DM and DL ───────────────────────────────────────
xd = (axes[0, N_COND * N_SL - 1].get_position().x1 +
      axes[0, N_COND * N_SL    ].get_position().x0) / 2
y0 = axes[-1, 0].get_position().y0
y1 = axes[ 0, 0].get_position().y1 + 0.05
fig.add_artist(plt.Line2D([xd, xd], [y0, y1], transform=fig.transFigure,
                          color='#4488CC', linewidth=1.2))

plt.savefig('results/figures/e11_all_params_conditions.png', dpi=300,
            bbox_inches='tight', facecolor='black')
plt.show()